In [39]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings

load_dotenv()

db_params = {
    "dbname": "data-sense-db",
    "user": os.getenv("POSTGRES_USERNAME"),
    "password": os.getenv("POSTGRES_PASSWORD"),
    "host": os.getenv("POSTGRES_HOST"),
    "port": "5432",
}
os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN")
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=os.getenv("GROQ_API_KEY"),
    temperature=0
)

llm.invoke("Hello")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 12078.54it/s]


AIMessage(content='Hello. How can I help you today?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 36, 'total_tokens': 46, 'completion_time': 0.027847349, 'completion_tokens_details': None, 'prompt_time': 0.001712352, 'prompt_tokens_details': None, 'queue_time': 0.160945917, 'total_time': 0.029559701}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_45180df409', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e630b-af34-7372-b61c-df5fd529db64-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 36, 'output_tokens': 10, 'total_tokens': 46})

In [40]:
import psycopg2
from langchain_core.prompts import ChatPromptTemplate
from pgvector.psycopg2 import register_vector
from collections import defaultdict

In [41]:
def retrieve_relevant_schema(question: str, top_k: int = 5) -> str:
    vec = embeddings.embed_query(question)
    conn = psycopg2.connect(**db_params)
    register_vector(conn)
    cur = conn.cursor()

    RETRIEVAL_SQL = """
        SELECT
            table_name,
            ROUND(
                MAX(
                    (1 - (embedding <=> %s::vector)) *
                    CASE chunk_type
                        WHEN 'use_cases' THEN 1.4
                        WHEN 'overview'  THEN 1.1
                        ELSE                  1.0
                    END
                )::numeric, 4
            ) AS weighted_similarity
        FROM schema_chunks
        GROUP BY table_name
        ORDER BY MAX(
            (1 - (embedding <=> %s::vector)) *
            CASE chunk_type
                WHEN 'use_cases' THEN 1.4
                WHEN 'overview'  THEN 1.1
                ELSE                  1.0
            END
        ) DESC
        LIMIT %s
    """
    cur.execute(RETRIEVAL_SQL, (vec, vec, top_k))
    top_tables = cur.fetchall()

    print("[RAG] Top tables by weighted similarity:")
    for table, score in top_tables:
        print(f"  {table}: {score}")

    table_names = [row[0] for row in top_tables]
    cur.execute(
        """
        SELECT table_name, chunk_type, ddl_chunk
        FROM schema_chunks
        WHERE table_name = ANY(%s)
        ORDER BY table_name, chunk_type
        """,
        (table_names,)
    )
    chunks = cur.fetchall()
    conn.close()

    table_chunks = defaultdict(list)
    for table, chunk_type, ddl_chunk in chunks:
        table_chunks[table].append(f"-- [{chunk_type}]\n{ddl_chunk}")

    return "\n\n".join(
        f"=== {table} ===\n" + "\n\n".join(table_chunks[table])
        for table in table_names  # preserve relevance order
    )

context = retrieve_relevant_schema("Which sellers in Mumbai have stock below reorder level for Fashion products?")
print("\n[Full Context]\n")
print(context)

[RAG] Top tables by weighted similarity:
  inventory: 0.7806
  sellers: 0.5251
  products: 0.4841
  orders: 0.4509
  order_items: 0.4492

[Full Context]

=== inventory ===
-- [columns]
TABLE: inventory
COLUMNS:
  inventory_id             character varying -- Unique inventory record identifier (PK)
  product_id               character varying -- FK → products.product_id — which product this stock belongs to
  seller_id                character varying -- FK → sellers.seller_id — which seller holds this stock
  warehouse                character varying -- City name of the fulfillment warehouse holding this stock [values: Mumbai | Delhi | Bengaluru | Hyderabad | Chennai | Kolkata | Pune | Ahmedabad]
  quantity_in_stock        integer -- Current number of units available for sale
  reorder_level            integer -- Stock threshold below which a reorder should be triggered
  reorder_quantity         integer -- Number of units to order when restocking
  last_restocked_at        date -- Da

In [42]:
SQL_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """
        Your are an expert PostgreSQL Engineer.
        Given the schema context below, write ONE optimized SQL Query
        that answers the user's question.
        Return only the SQL - no markdown fences, no explaination.
        SCHEMA:
        {schema_context}
        RECENT CONVERSATIONS:
        {conversation_history}
        Rules:
        - Never SELECT *
        - Always alias aggregations: SUM(x) AS total_x
        - Use LIMIT 100 unless user asks for everything
        - Never use INSERT / UPDATE / DELETE / TRUNCATE / DROP
        - Use date_trunc for time grouping
    """),
    ("human", "Question: {question}")
])

In [43]:
question = "Top 5 customers by total orders"
schema_context = retrieve_relevant_schema(question)
print(schema_context)

[RAG] Top tables by weighted similarity:
  order_items: 0.5896
  customers: 0.5857
  orders: 0.5120
  inventory: 0.4895
  order_payments: 0.4801
=== order_items ===
-- [columns]
TABLE: order_items
COLUMNS:
  order_item_id            character varying -- Unique line item identifier (PK). Format: OI000000001
  order_id                 character varying -- FK → orders.order_id — which order this item belongs to
  product_id               character varying -- FK → products.product_id — which product was purchased
  seller_id                character varying -- FK → sellers.seller_id — which seller fulfilled this item
  quantity                 smallint -- Number of units of this product in the order [values: 1 | 2 | 3 | 4 | 5  -- units of this product in the order]
  unit_price               numeric -- Price per unit at time of purchase (selling_price snapshot)
  mrp                      numeric -- MRP at time of purchase (for discount calculation)
  discount_percent         smallint -- Di

In [44]:
sql_generate_chain = SQL_PROMPT | llm

In [45]:
generated_sql = sql_generate_chain.invoke({
    "question": question,
    "schema_context": schema_context,
    "conversation_history": []
}).content.strip()

In [46]:
generated_sql

'SELECT c.customer_id, c.first_name, c.last_name, COUNT(o.order_id) AS total_orders \nFROM customers c \nJOIN orders o ON c.customer_id = o.customer_id \nGROUP BY c.customer_id, c.first_name, c.last_name \nORDER BY total_orders DESC \nLIMIT 5;'

In [47]:
import pandas as pd

def execute_sql(query: str):
    try:
        conn = psycopg2.connect(**db_params)
        cur = conn.cursor()
        cur.execute(query)
        cols = [d[0] for d in cur.description]
        rows = [dict(zip(cols, row)) for row in cur.fetchall()]
        conn.close()
        return rows, None
    except Exception as e:
        return None, str(e)

In [48]:
rows, error = execute_sql(generated_sql)

if error:
    print("SQL Error : ", error)
else:
    df = pd.DataFrame(rows)
    print(f"{len(df)} rows returned")
    print(df)

5 rows returned
  customer_id first_name last_name  total_orders
0  CUST003626      Rohan    Bansal            61
1  CUST007134      Palak    Pandey            59
2  CUST000588     Ishita    Bansal            58
3  CUST000473     Aditya     Arora            57
4  CUST001370      Megha     Mehta            56


In [49]:
EASY = "How many products are listed in the Electronics category?"
MEDIUM = "What is the total revenue generated by each seller?"
HARD = "Which sellers in Mumbai have stock below reorder level for Fashion products?"

In [50]:
schema_context = retrieve_relevant_schema(EASY)
generated_sql = sql_generate_chain.invoke({
    "question": EASY,
    "schema_context": schema_context,
    "conversation_history": []
}).content.strip()

print(generated_sql)

rows, error = execute_sql(generated_sql)
if error:
    print("SQL Error : ", error)
else:
    df = pd.DataFrame(rows)
    print(f"{len(df)} rows returned")
    print(df)

[RAG] Top tables by weighted similarity:
  products: 0.6868
  order_items: 0.4344
  inventory: 0.3897
  sellers: 0.3584
  orders: 0.2697
SELECT COUNT(product_id) AS total_electronics_products 
FROM products 
WHERE category = 'Electronics' 
LIMIT 100;
1 rows returned
   total_electronics_products
0                        1947


In [51]:
schema_context = retrieve_relevant_schema(MEDIUM)
generated_sql = sql_generate_chain.invoke({
    "question": MEDIUM,
    "schema_context": schema_context,
    "conversation_history": []
}).content.strip()

print(generated_sql)

rows, error = execute_sql(generated_sql)
if error:
    print("SQL Error : ", error)
else:
    df = pd.DataFrame(rows)
    print(f"{len(df)} rows returned")
    print(df)

[RAG] Top tables by weighted similarity:
  order_items: 0.5618
  sellers: 0.5121
  products: 0.4491
  orders: 0.3400
  customers: 0.3273
SELECT 
    oi.seller_id, 
    SUM(oi.subtotal + oi.gst_amount) AS total_revenue
FROM 
    order_items oi
WHERE 
    oi.is_returned = FALSE
GROUP BY 
    oi.seller_id
ORDER BY 
    total_revenue DESC
LIMIT 100;
100 rows returned
    seller_id total_revenue
0   SELL00952    3092630.44
1   SELL00275    3059085.81
2   SELL01393    2735100.06
3   SELL01276    2723947.32
4   SELL01143    2715534.57
..        ...           ...
95  SELL00992    2240584.40
96  SELL01105    2237942.36
97  SELL01521    2234150.92
98  SELL00817    2232837.59
99  SELL01522    2230500.52

[100 rows x 2 columns]


In [52]:
schema_context = retrieve_relevant_schema(HARD)
generated_sql = sql_generate_chain.invoke({
    "question": HARD,
    "schema_context": schema_context,
    "conversation_history": []
}).content.strip()

print(generated_sql)

rows, error = execute_sql(generated_sql)
if error:
    print("SQL Error : ", error)
else:
    df = pd.DataFrame(rows)
    print(f"{len(df)} rows returned")
    print(df)

[RAG] Top tables by weighted similarity:
  inventory: 0.7806
  sellers: 0.5251
  products: 0.4841
  orders: 0.4509
  order_items: 0.4492
SELECT DISTINCT s.seller_id, s.seller_name 
FROM sellers s 
JOIN inventory i ON s.seller_id = i.seller_id 
JOIN products p ON i.product_id = p.product_id 
WHERE s.city = 'Mumbai' AND p.category = 'Fashion' AND i.quantity_in_stock < i.reorder_level 
LIMIT 100;
17 rows returned
    seller_id           seller_name
0   SELL00027    Pathak Enterprises
1   SELL00059      Krishnan Traders
2   SELL00364        Pillai Traders
3   SELL00416    Bhatia Enterprises
4   SELL00586          Shah Traders
5   SELL00616              Shop2159
6   SELL00675        Pathak Traders
7   SELL00756            Prime Zone
8   SELL01075    Bansal Enterprises
9   SELL01090         Kumar Traders
10  SELL01346              Shop1881
11  SELL01347  Krishnan Enterprises
12  SELL01392     Mumbai Wholesaler
13  SELL01404            Smart Zone
14  SELL01460         Naidu Traders
15  SELL01